# EEG Model Optimization - Colab

**Optimized for speed and minimal RAM usage:**
- Always uses balanced class weights (no optimization needed)
- Aggressive memory cleanup after each model
- Fast parameter search with proven parameter ranges

**Models included:** EEGNet, CBraMod, CSP, Feature-based (RF/SVM), Eye-tracking (RF/SVM)

In [ ]:
# Mount Google Drive and install packages
from google.colab import drive
drive.mount('/content/drive')

# Install required packages (optimized list)
!pip install -q torch torchvision torchaudio
!pip install -q scikit-learn pandas numpy matplotlib seaborn
!pip install -q mne einops  # einops needed for CBraMod

# Verify GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️  No GPU available - models will run on CPU (slower)")

In [ ]:
# Navigate to repository and configure paths
import os
import sys
from pathlib import Path

# Update this path to match your Drive setup
%cd /content/drive/MyDrive/eye-eeg/eye-eeg
print(f"Repository directory: {os.getcwd()}")

In [ ]:
PARAM_GRIDS = {
    'eegnet': {
        'D': [2, 4],
        'F1': [8, 16],
        'F2': [16],
        'dropoutRate': [0.3],
        'kernLength': [64],
        'lr': [0.001, 0.01],
        'epochs': [75]
    },
    'cbramod': {
        'dropout': [0.1, 0.2],
        'hidden_dim': [256, 512],
        'label_smoothing': [0.0],
        'lr': [0.001, 0.005],
    },
    'csp': {
        'n_components': [2, 4, 6],
        'reg': [0.01, 0.05, 0.1],
        'clf_type': ['lda', 'rf']
    },
    'features_rf': {
        'n_estimators': [100, 200],
        'max_depth': [20, 30],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2]
    },
    'features_svm': {
        'C': [0.01, 0.1, 1.0],
        'kernel': ['rbf', 'linear'],
        'gamma': ['scale', 'auto']
    },
    'et_rf': {
        'n_estimators': [50, 100, 200],
        'max_depth': [20, 30],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2]
    },
    'et_svm': {
        'C': [0.01, 0.1, 1.0],
        'kernel': ['rbf', 'linear'],
        'gamma': ['scale', 'auto']
    }
}

In [ ]:
from classification.training_utils import optimize_single_model
from data.data_configs import SUBJECTS
from data.data_loader import DataLoader

DataLoader.data_path = Path("/content/drive/MyDrive/eye-eeg/data")

for model in PARAM_GRIDS.keys():
    print(f"Optimizing {model}...")
    optimize_single_model(model, SUBJECTS[:1], PARAM_GRIDS[model], max_trials=1)